# Practice 2 — Data contract and preprocessing (both challenges): R1 draft

**Role:** R1 — Data and preprocessing steward  
**Scope:** P2.1 (data contract and exploration, both challenges), the shared preprocessing blocks used by R2/R3/R4, and `features.md`.

> Draft notebook. Its sections are copied unchanged into `notebooks/P2_cyber.ipynb` and `notebooks/P2_storage.ipynb` by R5.

In [3]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


## 0. Setup

In [7]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Data paths, relative to the notebook location
DATA_DIR = "../../datos"
CYBER_DIR = DATA_DIR + "/cyber"
STORAGE_DIR = DATA_DIR + "/storage"

# Identifier, group and fold columns: never used as model inputs
FORBIDDEN = ["row_id", "feature_group", "config_group", "fold_id"]

## 1. Provenance

Read `datos/cyber/PROVENANCE.md` and `datos/storage/PROVENANCE.md`. Summarise: source, authority, licence, and what the course removed from each dataset and why (asked at the defence).

_Summary:_ TODO

# Challenge A — cyber

## 2. Load the contract tables

In [5]:
# --- Challenge A: load development table, groups and class codes ---
# Main table: one row per network connection, with its features and class
cyber = pd.read_csv(CYBER_DIR + "/cyber_development.csv")
# Groups file: tells which group each row belongs to (for validation)
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")
# Class codes: dictionary that maps each code to its name
with open("../../datos/cyber/cyber_class_codes.json") as fh:
    CLASS_NAMES = {int(k): v for k, v in json.load(fh).items()}

# shape = (rows, columns). Both files should have the same number of rows
print("cyber:", cyber.shape, "| groups:", cyber_groups.shape)
print("class codes:", CLASS_NAMES)

cyber.head()

cyber: (34928, 13) | groups: (34928, 2)
class codes: {0: 'backdoor', 1: 'ddos', 2: 'dos', 3: 'injection', 4: 'mitm', 5: 'normal', 6: 'password', 7: 'ransomware', 8: 'scanning', 9: 'xss'}


,row_id,proto,service,conn_state,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,dst_ip_bytes,attack_code
0,C00000,udp,dns,SF,0.001593,37,53,0,1,65,1,81,2
1,C00001,tcp,-,SH,4.036948,0,0,0,3,156,0,0,9
2,C00002,tcp,http,S3,5.165641,1867,1058,0,6,2187,4,1274,9
3,C00003,tcp,http,SF,0.013618,167,2987,0,5,435,7,3359,6
4,C00004,udp,dns,SF,0.017344,58,58,0,1,86,1,86,1


## 3. Verify the schema against the handout

Expected: `row_id`, 11 model features (3 nominal + 8 numeric), `attack_code`. Check the columns, their types, missing values, and that `row_id` matches `cyber_groups.csv`.

In [ ]:
CYBER_NOMINAL = ["proto", "service", "conn_state"]
CYBER_NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
                 "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]

# Model input columns: the 11 predictors
CYBER_FEATURES = CYBER_NOMINAL + CYBER_NUMERIC

# --- Challenge A: integrity checks ---
# Goal: make sure the data is clean before using it

# 1. Schema against the handout: row_id + 11 predictors + integer target 
expected_columns = ["row_id"] + CYBER_FEATURES + ["attack_code"]
print("columns as in the handout:", list(cyber.columns) == expected_columns)
print("number of predictors:", len(CYBER_FEATURES), "(expected 11)")

# No identifier, group or fold column may be used as a predictor 
for col in FORBIDDEN:
    print(col, "used as predictor:", col in CYBER_FEATURES, "(expected False)")

# 2. Target: integer type and the same codes as the JSON
print("\ntarget type:", cyber["attack_code"].dtype)
codes_in_data = sorted(cyber["attack_code"].unique().tolist())
codes_in_json = sorted(CLASS_NAMES.keys())
print("codes in data:", codes_in_data)
print("codes in JSON:", codes_in_json)

# Data type of each column
print()
print(cyber.dtypes)

# 3. Both files must list the rows in the same order, because later we attach each row to its group by position
print("\nrow_id aligned with groups file:", cyber["row_id"].equals(cyber_groups["row_id"]))

# 4. Every row must have a unique identifier
print("duplicated row_id:", cyber["row_id"].duplicated().sum(), "(expected 0)")

# 5. Missing values: first sum counts per column, second sum adds all columns
print("missing values:", cyber.isna().sum().sum(), "(expected 0)")

# 6. Range of each numeric predictor. The minimum must be >= 0 so that log1p is defined
print()
for col in CYBER_NUMERIC:
    values = cyber[col]
    positive_values = values[values>0]
    print(col, "| min:", values.min(), "| min non-zero:", positive_values.min(), "| max:", values.max())

# 7. Categories in the text columns and how often each one appears
for col in CYBER_NOMINAL:
    print()
    print(cyber[col].value_counts())

_Interpretation:_ 

**Schema:** 34.928 rows x 13 columns, in the handout order: `row_id` (identifier only, never a predictor), 3 nominal predictors read as text (`proto`, `service`, `conn_state`), 8 numeric predictors (`duration` as float, the 7 counts as integers) and the integer target `attack_code`. Its 10 codes are exactly the ones in the JSON. No identifier, group or fold column is in the predictor list

**Integrity:** groups file aligned row by row; no duplicated `row_id`, no missing values; all numeric predictors >= 0, so `log1p` is defined

**Scales:** the byte counts go from a smallest non-zero value of 1 up to about 1.9x10⁹, and `duration` goes from 10⁻⁶ to 9.3x10⁴. On the raw scale a Euclidean distance would be dominated by the byte counts

**Rare categories:** `service=gssapi` has a single row, `dce_rpc` 7 and `smb;gssapi` 15; `conn_state = RSTOS0` has 32 and `proto = icmp` 74. A category with one row is necessarily absent from the training portion of the fold where it is validated, so the encoder must be fitted inside each fold with `handle_unknown="ignore"`

## 4. Class supports

Report the support of each class with its semantic name; name **ransomware's** support explicitly.

In [ ]:
# --- Challenge A: class distribution ---
# 1. Number of rows per class, ordered by code
counts = cyber["attack_code"].value_counts().sort_index()

print("code | name | rows | % of total")
for code in counts.index:
    n_rows = counts[code]
    pct = 100 * n_rows /len(cyber)
    print(code, "|", CLASS_NAMES[code], "|", n_rows, "|", round(pct, 2))

# 2. Ransomware: find its code by name
for code, name in CLASS_NAMES. items():
    if name == "ransomware":
        RANSOMWARE = code

n_ransomware = counts[RANSOMWARE]
rows_per_fold = n_ransomware / 5
one_row_in_recall = 100 / rows_per_fold

print("\nransomware rows:", n_ransomware)
print("ransomware rows per fold:", round(rows_per_fold))
print("one row = recall points:", round(one_row_in_recall, 1))

_Interpretation:_ 

**Imbalance:** six classes (`ddos`, `dos`, `injection`, `normal`, `password`, `xss`) have 4.800 rows each (13.74%), while `scanning` has 3.542, `backdoor` 1.573, `mitm` 805 and ransomware only 208 (0.60%). With 5 folds that is about 42 ransomware rows per fold, so one row changes its fold recall by ~2.4 points. Accuracy alone is not enough: report per-class recall with its spread across folds. The six classes with 4.800 rows are the ones capped at 6.000 by the course. As PROVENANCE requires, every per-class figure for ransomware must be reported beside its support (208)

## 5. Feature groups

Number of distinct `feature_group` values, and how many groups contain more than one row (exact feature-vector duplicates).

In [ ]:
# --- Challenge A: rows sharing a feature_group ---
# Are rows that share a group exact copies? Do they have the same class?

# 1. Copy of the data with the group column added
# Adding it by position is safe because section 3 confirmed the row order matches. The original `cyber` table never gets the group column
cyber_with_group = cyber.copy()
cyber_with_group["feature_group"] = cyber_groups["feature_group"].values

# 2. How many rows each group has
rows_per_group = cyber_with_group["feature_group"].value_counts()
print("number of groups:", len(rows_per_group))
print("max rows per group:", rows_per_group.max())

# 3. Keep only the rows of groups with 2 or more rows
multi_row_groups = rows_per_group[rows_per_group > 1].index
dup = cyber_with_group[cyber_with_group["feature_group"].isin(multi_row_groups)]
print("groups with more than one row:", len(multi_row_groups))
print("rows in those groups:", len(dup))

# 4. Go through those groups one by one
n_identical = 0
n_mixed_labels = 0
n_all_labels_differ = 0
unavoidable_errors = 0
conflicting_groups = []

for group_id, rows in dup.groupby("feature_group"):
    # Exact copies: only one distinct feature vector inside the group
    n_distinct_vectors = len(rows[CYBER_FEATURES].drop_duplicates())
    if n_distinct_vectors == 1:
        n_identical += 1

    # Labels inside the group
    n_labels = rows["attack_code"].nunique()
    if n_labels > 1:
        n_mixed_labels += 1
        conflicting_groups.append(group_id)
    if n_labels == len(rows):
        n_all_labels_differ += 1

    # For identical rows, the best a model can do is predict the most frequent class, so the remaining rows of the group are errors whatever the model
    most_frequent_count = rows["attack_code"].value_counts().max()
    unavoidable_errors += len(rows) - most_frequent_count

print("groups with identical features:", n_identical)
print("groups with more than one class:", n_mixed_labels)
print("groups where every copy has a different label:", n_all_labels_differ)

# 5. Reverse check: identical feature vectors never sit in different groups. If so, the number of distinct feature vectors equals the number of groups
n_distinct_vectors_total = len(cyber[CYBER_FEATURES].drop_duplicates())
print("distinct feature vectors:", n_distinct_vectors_total, "| groups:", len(rows_per_group))

# 6. Show the conflicting rows with their class name
conflict_rows = dup[dup["feature_group"].isin(conflicting_groups)]
conflict_rows = conflict_rows[["feature_group", "attack_code"]].copy()
conflict_rows["name"] = conflict_rows["attack_code"].map(CLASS_NAMES)
print(conflict_rows.sort_values("feature_group"))

# 7. Maximim accuracy any model can reach
max_accuracy = 100 * (len(cyber) - unavoidable_errors) / len(cyber)
print("unavoidable errors (minimum):", unavoidable_errors)
print("maximum accuracy (%):", round(max_accuracy, 2))

_Interpretation:_ 

**Feature groups:** 38 of the 34.883 groups have more than one row (83 rows, at most 4 per group). Rows in the same group are exact copies, and identical feature vectors never sit in different groups, so each `feature_group` is exactly one distinct feature vector. Grouped validation keeps every copy on the same side

**Label conflicts:** all 38 groups mix classes, ransomware included; in fact every copy inside a group carries a different label. At least 45 errors are unavoidable: maximum accuracy (34.928 - 45) / 34.928 = 99.87%. The rows are kept: the modelling table is fixed

**Why every copy has a different label:** the course removed exact duplicates of `(X,y)` before distributing the data. Two rows with the same features and the same class were kept once, so the only copies left are those whose labels differ. This also explains the 45: with every label different, the best a model can do is one correct row per group, so 
83-38=45 error

## 6. Preprocessing block (cyber)

One-hot on the 3 nominal predictors; `log1p` then standardisation on the 8 numeric predictors. It is returned unfitted, to be placed inside a `Pipeline` so that it is fitted inside each fold. **Publish this cell for R2 and R4.**

In [ ]:
def make_cyber_preprocessor():
    numeric_steps = Pipeline([
        ("log1p", FunctionTransformer(np.log1p)),
        ("scake", StandardScaler())
    ])

    nominal_steps = OneHotEncoder(handle_unknown="ignore")

    preprocessor = ColumnTransformer([
        ("nominal", nominal_steps, CYBER_NOMINAL),
        ("numeric", numeric_steps, CYBER_NUMERIC),
    ])
    return preprocessor

_Why `log1p` and why scaling:_ 

**`log1p`:** the numeric predictors are counts and durations spanning up to nine orders of magnitude, with many zeros. `log` is undefined at 0, while `log1p(x) = log(1 + x)` maps 0 to 0 and is defined because every value is >= 0. It compresses the range: 1.4x10⁹ bytes becomes about 21, so a few huge connections no longer dominate

**Scaling:** after `log1p` the columns still have different ranges (`duration` stays below about 11.4, the byte counts reach about 21). `StandardScaler` puts every numeric column on mean 0 and standard deviation 1, so no predictor dominates a distance just because of its units

**Inside each fold:** the block is returned unfitted. Inside a `Pipeline`, the means, standard deviations and one-hot categories are learned only from the training portion of each fold, so no information from the validation rows leaks into de model

In [ ]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Data paths, relative to the notebook location
DATA_DIR = "../../datos"
CYBER_DIR = DATA_DIR + "/cyber"
STORAGE_DIR = DATA_DIR + "/storage"

# Identifier, group and fold columns: never used as model inputs
FORBIDDEN = ["row_id", "feature_group", "config_group", "fold_id"]

# Challenge B — storage

## 7. Load the contract tables

In [ ]:
storage = pd.read_csv("../../datos/storage/storage_development.csv")
storage_groups = pd.read_csv("../../datos/storage/storage_groups.csv")

storage.head()

## 8. Verify the schema against the handout

Expected: `row_id`, 9 configuration/workload features (4 nominal + 5 numeric), `latency_ns`. `iops` must **not** be present. Note: `raid` looks numeric (`4+2`) but is **nominal**.

In [1]:
STORAGE_NOMINAL = ["load_type", "io_type", "raid", "device_type"]
STORAGE_NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]

STORAGE_FEATURES = STORAGE_NOMINAL + STORAGE_NUMERIC

# --- Challenge B: integrity checks and value ranges ---

# 1. Schema against the handout: row_id + 9 predictors + latency_ns, and no iops
# The file uses a different column order, so we compare the names sorted alphabetically
expected_columns = ["row_id"] + STORAGE_FEATURES + ["latency_ns"]
print("number of columns:", storage.shape[1], "(expected 11)")
print("same columns as handout:", sorted(storage.columns) == sorted(expected_columns))
print("number of predictors:", len(STORAGE_FEATURES), "(expected 9)")
print("iops present:", "iops" in storage.columns, "(expected False)")

# No identifier, group or fold column may be used as a predictor
for col in FORBIDDEN:
    print(col, "used as predictor:", col in STORAGE_FEATURES, "(expected False)")

# Data tupe of each column
print()
print(storage.dtypes)

# 2. Both files must list the rows in the same order
print("\nrow_id aligned with groups file:", storage["row_id"].equals(storage_groups["row_id"]))

# 3. Duplicated row_ide values and missing values
print("duplicated row_id:", storage["row_id"].duplicated().sum(), "(expected 0)")
print("missing values:", storage.isna().sum().sum(), "(expected 0)")

# 4. Distinct values of each input variable, sorted
for col in STORAGE_NUMERIC + STORAGE_NOMINAL:
    print()
    print(col, ":", sorted(storage[col].unique().tolist()))

# 5. Target: summary, finite values, strictly positive, and how skewed it is
lat = storage["latency_ns"]
print("\nlatency_ns:")
print(lat.describe())
print("all values finite:", np.isfinite(lat).all())
print("minimum latency:", lat.min(), "(must be > 0)")

ratio_max_min = lat.max() / lat.min()
orders_of_magnitude = np.log10(ratio_max_min)
mean_over_median = lat.mean() / lat.median()
print("orders of magnitude:", round(orders_of_magnitude, 1))
print("mean / median:", round(mean_over_median, 1))

NameError: name 'storage' is not defined

_Interpretation:_ 

**Schema:** 12.190 rows x 11 columns: `row_id`, 5 integer and 4 nominal predictors, and the target `latency_ns`. The file uses a different column order from the handout, so the names are compared sorted. `raid` is read as text, not as a number. `iops` is absent, as rule 4 requires: it is a second output of the same run, tied to latency, so it would be a shortcut. No identifier, group or fold column is in the predictor list

**Integrity:** groups file aligned row by row; no duplicated `row_id`, no missing values; the target is finite and strictly positive

**Scales:** predictors lie on discrete grids with very different ranges (`iodepth` 1-32, `block_size` 4-1024), so distance-based methods need standardisation inside each fold

**Target:** from 1.16x10⁵ to 2.53x10⁹ ns; the mean is 23 times the median, so the distribution has a long right tail. 

## 9. Configurations and replicates

Number of distinct `config_group` values and the replicate count per configuration.

In [ ]:
# --- Challenge B: configurations and replicates ---

# 1. Copy of the data with the group column added
# The original `storage` table never gets the group column
storage_with_group = storage.copy()
storage_with_group["config_group"] = storage_groups["config_group"].values

# 2. Number of groups and number of rows per group
rows_per_group = storage_with_group["config_group"].value_counts()
print("number of groups:", len(rows_per_group))
print("smallest group:", rows_per_group.min(), "rows")
print("largest group:", rows_per_group.max(), "rows")

# 3. The replicates of a group must share all 9 features
groups_with_differences = 0
for group_id, rows in storage_with_group.groupby("config_group"):
    n_distinct_configs = len(rows[STORAGE_FEATURES].drop_duplicates())
    if n_distinct_configs > 1:
        groups_with_differences += 1
print("groups whose replicates differ:", groups_with_differences, "(expected 0)")

# 4. Reverse check: the same configuration never appears in two different groups
# If so, the number of distinct configurations equals the number of groups 
n_distinct_configs_total = len(storage[STORAGE_FEATURES].drop_duplicates())
print("distinct configurations:", n_distinct_configs_total, "| groups:", len(rows_per_group))

_Interpretation:_ 

**Configuration groups:** 2.438 groups of exactly 5 rows. The 5 replicates of a group share all 9 predictors, and the number of distinct configurations equals the number of groups, so no configuration is split across groups: each `config_group` is exactly one configuration. The unit of generalisation is the configuration, so validation must keep the 5 replicates together. In the source each configuration was measured 60-180 times; the course kept 5 because scores are flat from five upward.

## 10. Preprocessing block (storage)

One-hot on the 4 nominal predictors; standardisation on the 5 numeric predictors. **Publish this cell for R3 and R4.**

In [ ]:
def make_storage_preprocessor():
    # Nominal (raid included): one column per category; unseen categories become all zeros
    nominal_steps = OneHotEncoder(handle_unknown="ignore")

    # Numeric: put every column on the same scale
    numeric_steps = StandardScaler()

    preprocessor = ColumnTransformer([
        ("nominal", nominal_steps, STORAGE_NOMINAL),
        ("numeric", numeric_steps, STORAGE_NUMERIC),
    ])
    return preprocessor

_Interpretation:_ 

**One-hot:** the 4 nominal predictors have no numeric meaning. `raid` looks numeric (`4+2`) but it is a label: `4+2` is a redundancy scheme, not a quantity: treating it as a number would assert an order and a spacing the data do not have, not quantities, so it is one-hot encoded like `load_type`, `io_type` and `device_type`. `handle_unknown="ignore"` turns a category unseen in training into all zeros instead of an error

**Scaling:** the 5 integer predictors have very different ranges (`iodepth` 1-32, `block_size` 4-1024, `read_fraction` 0-100), so `StandardScaler` puts them on the same scale. It is fitted inside each fold, like the cyber block

**Target:** this block only transforms the predictors. `latency_ns` is not transformed here.

## 11. `features.md`

Fill in `features.md` at the repo root from `plantillas/features.md` (copy it, do not edit the template): one dated row per source feature, plus the `log1p` representation (cyber) and the `z = log10(latency_ns)` target representation (storage, after R3's P2.3).